# 06 — Multi-Label Classification Basics

**Objective:** understand what actually changes when you go from single-label (CTI 101 notebook 08) to multi-label classification — *before* we touch TRAM data in notebook 07.

## Why a whole notebook on this?

Multi-label looks like a small tweak ("just allow more than one label") but it changes four things at once:

1. The **label format** — one integer per sample becomes a binary vector.
2. The **activation function** — softmax becomes sigmoid.
3. The **loss function** — cross-entropy becomes binary cross-entropy.
4. The **evaluation metrics** — accuracy is nearly useless; you need Hamming loss, per-label F1, and threshold tuning.

Skipping this stuff and jumping straight to notebook 07 means debugging all four at once when something goes wrong. Way easier to see them in isolation on a synthetic example first.

## What this notebook does

1. Contrast single-label and multi-label setups side by side
2. Build a tiny synthetic dataset you can reason about completely
3. Train a small model with the right loss / activation
4. Tune decision thresholds per label
5. Report all the multi-label metrics and explain what each one is telling you

No BERT, no TRAM, no GPUs. A toy dataset and a linear model — the point is pedagogy, not accuracy.

## Step 1 — Single-label vs multi-label, side by side

Take the sentence *"The attacker phished credentials and used them to move laterally."*

### Single-label setup (CTI 101 style)

- Assumes every sentence has **exactly one** correct class.
- Label format: `2` (an integer — say, class index for "initial-access").
- Model outputs logits of shape `[num_classes]`.
- Apply **softmax** → probabilities sum to 1 across classes. Pick the highest.
- Loss: **cross-entropy** — penalizes everything except the one correct class.
- Metric: accuracy. Simple.

The assumption "exactly one class per sample" is baked in *everywhere*, from the softmax (classes compete for probability mass) to the loss (there can only be one correct answer) to accuracy (the prediction is either right or wrong).

### Multi-label setup

- Samples can have **zero, one, or many** labels, independently.
- Label format: `[0, 0, 1, 0, 1, 0, 0, ...]` — a multi-hot vector of length `num_classes`, with 1s at positive positions.
- Model outputs logits of shape `[num_classes]`.
- Apply **sigmoid to each output independently** — each logit becomes an independent probability in `[0, 1]`. They do **not** sum to 1.
- Loss: **binary cross-entropy** — treats the problem as `num_classes` independent yes/no questions. For each class: "is this label present?"
- Metrics: subset accuracy is too strict, plain accuracy is misleading. Use Hamming loss, micro/macro F1, and tune per-label thresholds.

The key mental shift: **each label is its own binary classifier**, wired into a shared model. Softmax is replaced by N independent sigmoids.

### Why not "just use softmax with multiple correct answers"?

Because softmax forces probabilities to sum to 1. If two labels are both correct, the best softmax can do is split probability 0.5/0.5 — which *looks* uncertain, even when the model is confident about both. Independent sigmoids let the model say 0.9 for both simultaneously, which is what we want.

## Step 2 — Build a synthetic toy dataset

A 3-label classification problem where every label depends on a different feature:

- Features: 10 floats per sample.
- **Label A** = 1 iff `feature[0] + feature[1] > 0`
- **Label B** = 1 iff `feature[2] - feature[3] > 0.5`
- **Label C** = 1 iff `feature[4]*feature[5] > 0`

These three rules are independent — a single sample can trigger any combination from 0 to 3 labels. Pure multi-label setup, and because we **know** the ground-truth rules we can tell whether the model learned anything real.

In [1]:
import numpy as np

rng = np.random.default_rng(42)
N = 2000
D = 10
NUM_LABELS = 3
LABEL_NAMES = ['A', 'B', 'C']

X = rng.standard_normal((N, D)).astype(np.float32)

Y = np.zeros((N, NUM_LABELS), dtype=np.float32)
Y[:, 0] = (X[:, 0] + X[:, 1] > 0).astype(np.float32)
Y[:, 1] = (X[:, 2] - X[:, 3] > 0.5).astype(np.float32)
Y[:, 2] = (X[:, 4] * X[:, 5] > 0).astype(np.float32)

# Split 80/20
split = int(0.8 * N)
X_tr, X_te = X[:split], X[split:]
Y_tr, Y_te = Y[:split], Y[split:]

print(f'Train: {X_tr.shape}, labels {Y_tr.shape}')
print(f'Test:  {X_te.shape}, labels {Y_te.shape}')

print(f'\nLabel prevalence in training set:')
for i, name in enumerate(LABEL_NAMES):
    print(f'  {name}: {Y_tr[:, i].mean():.3f}')

print(f'\nNumber of active labels per sample (train):')
active_counts = Y_tr.sum(axis=1).astype(int)
for k in range(NUM_LABELS + 1):
    pct = (active_counts == k).mean() * 100
    print(f'  {k} labels: {pct:.1f}%')

Train: (1600, 10), labels (1600, 3)
Test:  (400, 10), labels (400, 3)

Label prevalence in training set:
  A: 0.514
  B: 0.359
  C: 0.516

Number of active labels per sample (train):
  0 labels: 14.6%
  1 labels: 41.6%
  2 labels: 34.0%
  3 labels: 9.8%


Notice that label prevalences are different (label B is rarer because its threshold is 0.5, not 0), and that many samples have 0 or multiple positive labels. Both are characteristic of real multi-label data — including TRAM.

## Step 3 — Model, loss, optimizer

A single linear layer with 3 outputs. This is the smallest thing that can do multi-label classification.

### The critical line

```python
loss_fn = nn.BCEWithLogitsLoss()
```

This is the multi-label workhorse. It does two things in one step:

1. Applies sigmoid to each of the 3 output logits independently.
2. Computes binary cross-entropy against each of the 3 target bits independently.
3. Sums/averages across the 3 labels.

The "`WithLogits`" suffix means you feed it **raw logits**, not sigmoid'd probabilities — it applies the sigmoid internally, which is more numerically stable than doing it yourself.

**What you must NOT do**: use `nn.CrossEntropyLoss()` (single-label) or apply softmax. Both enforce a sum-to-one constraint that's wrong for multi-label.

In [2]:
import torch
import torch.nn as nn

torch.manual_seed(42)

model = nn.Linear(D, NUM_LABELS)
loss_fn = nn.BCEWithLogitsLoss()
optim = torch.optim.Adam(model.parameters(), lr=0.05)

X_tr_t = torch.tensor(X_tr)
Y_tr_t = torch.tensor(Y_tr)
X_te_t = torch.tensor(X_te)

for epoch in range(200):
    model.train()
    logits = model(X_tr_t)
    loss = loss_fn(logits, Y_tr_t)
    optim.zero_grad()
    loss.backward()
    optim.step()
    if (epoch + 1) % 50 == 0:
        print(f'  epoch {epoch+1:3d}   loss {loss.item():.4f}')

  epoch  50   loss 0.3624
  epoch 100   loss 0.3252
  epoch 150   loss 0.3083
  epoch 200   loss 0.2979


## Step 4 — From logits to predictions

At inference:

1. Run the model → logits (shape `[N, 3]`).
2. Apply **sigmoid** element-wise → independent probabilities in `[0, 1]`.
3. **Threshold** each probability (default 0.5) to get a binary prediction.

The threshold is a hyperparameter — often the biggest lever for lifting F1 on rare labels.

In [3]:
model.eval()
with torch.no_grad():
    logits_te = model(X_te_t)
    probs_te = torch.sigmoid(logits_te).numpy()

preds_te = (probs_te >= 0.5).astype(np.float32)

print(f'First 5 test samples:')
print(f'{"truth":12s}  {"probs":24s}  {"pred @0.5":12s}')
for i in range(5):
    truth = Y_te[i].astype(int).tolist()
    prob = [f'{p:.2f}' for p in probs_te[i]]
    pred = preds_te[i].astype(int).tolist()
    print(f'  {str(truth):12s}  [{", ".join(prob)}]  {str(pred):12s}')

First 5 test samples:
truth         probs                     pred @0.5   
  [1, 0, 0]     [0.61, 0.05, 0.58]  [1, 0, 1]   
  [0, 0, 1]     [0.00, 0.32, 0.52]  [0, 0, 1]   
  [0, 0, 0]     [0.26, 0.00, 0.57]  [0, 0, 1]   
  [1, 0, 0]     [0.99, 0.00, 0.50]  [1, 0, 1]   
  [0, 0, 1]     [0.01, 0.00, 0.58]  [0, 0, 1]   


## Step 5 — Evaluation metrics (and why plain accuracy is misleading)

We'll look at five numbers, because no single one tells the whole story.

### Subset accuracy (exact-match accuracy)

Fraction of test samples where **every single label** was predicted correctly. The strictest metric — predicting 2 of 3 labels right counts as zero.

### Hamming loss / Hamming accuracy

Fraction of **individual label decisions** that are wrong (or right). It treats the `N × L` decision matrix as a flat vector of yes/no answers. Much more forgiving than subset accuracy.

### Per-label F1

Compute precision/recall/F1 for each label independently, treating "is label-i present?" as its own binary problem.

### Micro-F1

Pool every (sample, label) decision across all labels, then compute one F1 on the pool. Dominated by frequent labels.

### Macro-F1

Average the per-label F1s. Rare labels count equally with common ones. Usually what you report when you care about **all** classes, including the tail.

In [4]:
from sklearn.metrics import (
    accuracy_score,            # subset accuracy when used on multi-label
    hamming_loss,
    precision_recall_fscore_support,
    f1_score,
)

subset_acc = accuracy_score(Y_te, preds_te)
h_loss = hamming_loss(Y_te, preds_te)
hamming_acc = 1.0 - h_loss

micro_p, micro_r, micro_f, _ = precision_recall_fscore_support(
    Y_te, preds_te, average='micro', zero_division=0
)
macro_p, macro_r, macro_f, _ = precision_recall_fscore_support(
    Y_te, preds_te, average='macro', zero_division=0
)

print(f'Subset (exact-match) accuracy: {subset_acc:.4f}')
print(f'Hamming accuracy (per-decision): {hamming_acc:.4f}')
print(f'Hamming loss (fraction wrong):   {h_loss:.4f}')
print()
print(f'Micro P/R/F1: {micro_p:.4f} / {micro_r:.4f} / {micro_f:.4f}')
print(f'Macro P/R/F1: {macro_p:.4f} / {macro_r:.4f} / {macro_f:.4f}')

print('\nPer-label:')
per_label = precision_recall_fscore_support(Y_te, preds_te, average=None, zero_division=0)
for i, name in enumerate(LABEL_NAMES):
    p, r, f, n = per_label[0][i], per_label[1][i], per_label[2][i], per_label[3][i]
    print(f'  {name}: P={p:.3f} R={r:.3f} F1={f:.3f}  (support={n})')

Subset (exact-match) accuracy: 0.5025
Hamming accuracy (per-decision): 0.8300
Hamming loss (fraction wrong):   0.1700

Micro P/R/F1: 0.8053 / 0.8502 / 0.8271
Macro P/R/F1: 0.8401 / 0.8659 / 0.8521

Per-label:
  A: P=0.986 R=0.991 F1=0.988  (support=213)
  B: P=1.000 R=0.993 F1=0.997  (support=146)
  C: P=0.534 R=0.614 F1=0.571  (support=215)


### Reading the numbers

- Subset accuracy is always lower than Hamming accuracy. The bigger the gap, the more "partial credit" the model is getting.
- Micro vs macro: if micro is much higher than macro, common labels are carrying you. That's exactly the situation in notebook 07 (a few tactics dominate TRAM).

## Step 6 — Threshold tuning

Using 0.5 as the decision threshold is a *convention*, not a law. For rare labels, 0.5 is often too high — the model's probabilities concentrate below 0.5 because the label is rare, so recall collapses.

**Per-label threshold tuning**: for each label, sweep thresholds across a validation set and pick the one that maximizes F1 for that label. Then use those thresholds at test time.

Here we use the test set for simplicity — in a real run (notebook 07) you tune on validation and evaluate on held-out test.

In [5]:
best_thresholds = np.zeros(NUM_LABELS)

thresholds_to_try = np.linspace(0.05, 0.95, 37)

print(f'{"label":6s}  {"threshold":>10s}  {"F1 @ best":>10s}  {"F1 @ 0.5":>10s}')
for i, name in enumerate(LABEL_NAMES):
    f1s = [f1_score(Y_te[:, i], (probs_te[:, i] >= t).astype(np.float32),
                    zero_division=0) for t in thresholds_to_try]
    best_idx = int(np.argmax(f1s))
    best_thresholds[i] = thresholds_to_try[best_idx]
    f1_default = f1_score(Y_te[:, i], (probs_te[:, i] >= 0.5).astype(np.float32),
                          zero_division=0)
    print(f'  {name:4s}  {thresholds_to_try[best_idx]:10.2f}  {f1s[best_idx]:10.4f}  {f1_default:10.4f}')

label    threshold   F1 @ best    F1 @ 0.5
  A           0.55      0.9906      0.9883
  B           0.50      0.9966      0.9966
  C           0.05      0.6992      0.5714


Compare "F1 @ best" vs "F1 @ 0.5" per label. If tuned thresholds give meaningful gains, they're worth the extra bookkeeping.

Now re-evaluate using the tuned thresholds.

In [6]:
preds_tuned = (probs_te >= best_thresholds).astype(np.float32)

micro_p2, micro_r2, micro_f2, _ = precision_recall_fscore_support(
    Y_te, preds_tuned, average='micro', zero_division=0
)
macro_p2, macro_r2, macro_f2, _ = precision_recall_fscore_support(
    Y_te, preds_tuned, average='macro', zero_division=0
)

print(f'{"metric":12s}  {"@0.5":>10s}  {"@tuned":>10s}  gain')
print('-' * 48)
print(f'{"subset acc":12s}  {subset_acc:10.4f}  {accuracy_score(Y_te, preds_tuned):10.4f}  {accuracy_score(Y_te, preds_tuned) - subset_acc:+.4f}')
print(f'{"hamming acc":12s}  {hamming_acc:10.4f}  {1-hamming_loss(Y_te, preds_tuned):10.4f}  {hamming_acc - hamming_loss(Y_te, preds_tuned) - (hamming_acc-1):+.4f}')
print(f'{"micro F1":12s}  {micro_f:10.4f}  {micro_f2:10.4f}  {micro_f2 - micro_f:+.4f}')
print(f'{"macro F1":12s}  {macro_f:10.4f}  {macro_f2:10.4f}  {macro_f2 - macro_f:+.4f}')

metric              @0.5      @tuned  gain
------------------------------------------------
subset acc        0.5025      0.5325  +0.0300
hamming acc       0.8300      0.8417  +0.8417
micro F1          0.8271      0.8571  +0.0300
macro F1          0.8521      0.8954  +0.0433


Macro F1 usually benefits most from per-label tuning, because rare labels are the ones where 0.5 is the wrong threshold.

## Step 7 — One common gotcha: label prevalence ≠ class imbalance

In single-label classification, "class imbalance" means one class dominates. In multi-label, there's a subtler issue: for every individual label, the vast majority of samples are **negative** for that label.

Even if label A fires 40% of the time (quite common), that still means each example is negative for the *other* labels most of the time. A model that just predicts all-zeros gets high Hamming accuracy.

Let's check.

In [7]:
all_zeros = np.zeros_like(Y_te)
print(f'Always-predict-zero baseline:')
print(f'  subset accuracy:  {accuracy_score(Y_te, all_zeros):.4f}')
print(f'  hamming accuracy: {1 - hamming_loss(Y_te, all_zeros):.4f}')
print(f'  macro F1:         {f1_score(Y_te, all_zeros, average="macro", zero_division=0):.4f}')
print(f'  micro F1:         {f1_score(Y_te, all_zeros, average="micro", zero_division=0):.4f}')

Always-predict-zero baseline:
  subset accuracy:  0.1475
  hamming accuracy: 0.5217
  macro F1:         0.0000
  micro F1:         0.0000


Hamming accuracy can be shockingly high just by predicting nothing. That's why macro F1 is usually the right headline number for multi-label — it refuses to give you credit for saying "no" to rare labels.

## Summary — the multi-label checklist

When you move to `BertForSequenceClassification` in notebook 07, these are the things that will be set correctly because of this notebook:

1. Labels are **multi-hot float vectors** of length `num_classes`, not single integers.
2. Model is configured with `problem_type="multi_label_classification"` — this makes HuggingFace use `BCEWithLogitsLoss` internally.
3. Predictions come from **sigmoid + threshold**, not argmax.
4. Per-label thresholds are tuned on validation, not left at 0.5.
5. **Macro F1** is the headline metric. Also report Hamming loss and per-label F1.

**Next:** notebook 07 applies exactly this recipe to TRAM, with BERT and 14 MITRE tactics replacing our toy 3-label setup.